# Transformation to SQL query

In [0]:
query = """
SELECT
    sd.order_number,
    pr.product_id,
    cu.customer_id,
    sd.order_date,
    sd.ship_date,
    sd.due_date,
    sd.sales_amount,
    sd.quantity,
    sd.unit_price
FROM silver.crm_sales_details sd
LEFT JOIN gold.dim_products pr
    ON sd.product_key = pr.product_key
LEFT JOIN gold.dim_customers cu
    ON sd.customer_id = cu.customer_id
"""
df = spark.sql(query)
 
df.limit(10).display()

# Duplicate order-product rows check

In [0]:
duplicate_grain_count = (
    df.groupBy("order_number", "product_id")
      .count()
      .filter("count > 1")
      .count()
)
 
if duplicate_grain_count > 0:
    raise Exception("Duplicate (order_number, product_id)")



#Write it to Gold Table

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.gold.fact_sales")


## Sanity check

In [0]:
%sql
SELECT * FROM gold.fact_sales LIMIT(10)

